In [1]:
!pip install -q streamlit groq onnx onnxruntime pyngrok torch onnxscript

In [ ]:
app_code = open('app.py').read() if False else """
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
import onnxruntime as ort
import io
from groq import Groq

class SimpleRouter(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(3, 3)
    def forward(self, x):
        return self.fc(x)

def get_keyword_scores(text):
    text = text.lower()
    science_words = ['data','research','technology','science','evidence','study','experiment']
    economy_words = ['money','market','cost','economy','profit','job','trade','price','growth']
    ethics_words  = ['right','wrong','moral','ethical','fair','should','harm','good','value']
    return np.array([
        sum(w in text for w in science_words),
        sum(w in text for w in economy_words),
        sum(w in text for w in ethics_words),
    ], dtype=np.float32)

@st.cache_resource(show_spinner=False)
def load_onnx_session():
    model = SimpleRouter()
    X = torch.eye(3)
    y = torch.tensor([0, 1, 2])
    opt = torch.optim.Adam(model.parameters(), lr=0.1)
    for _ in range(200):
        opt.zero_grad()
        nn.CrossEntropyLoss()(model(X), y).backward()
        opt.step()
    buf = io.BytesIO()
    torch.onnx.export(model, torch.zeros(1, 3), buf, input_names=['scores'], output_names=['logits'], opset_version=11)
    buf.seek(0)
    return ort.InferenceSession(buf.read())

def rank_perspectives(sess, query):
    scores = get_keyword_scores(query).reshape(1, -1)
    logits = sess.run(['logits'], {'scores': scores})[0][0]
    probs  = torch.softmax(torch.tensor(logits), dim=0).numpy()
    names  = ['Scientist', 'Economist', 'Ethicist']
    return sorted(zip(names, probs), key=lambda x: -x[1])

def ask_groq(client, question, perspective):
    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {'role': 'system', 'content': f'You are a {perspective}. Answer the question from a {perspective.lower()} point of view in 3-4 sentences. Be direct and use your field language.'},
            {'role': 'user',   'content': question},
        ],
        max_tokens=250, temperature=0.7,
    )
    return resp.choices[0].message.content.strip()

st.set_page_config(page_title='MultiMind AI', page_icon='🧠', layout='centered')
st.markdown('<style>body{background:#0f0f14;color:#e0dff0;font-family:sans-serif;}.card{padding:1.2rem;border-radius:12px;margin-bottom:1rem;border-left:5px solid;background:#1a1a24;}.label{font-size:0.78rem;font-weight:700;letter-spacing:2px;text-transform:uppercase;margin-bottom:0.5rem;}.body{font-size:0.95rem;color:#bbb;line-height:1.7;}</style>', unsafe_allow_html=True)

st.title('🧠 MultiMind AI')
st.caption('One question — answered by a Scientist, Economist, and Ethicist.')

question = st.text_input('Ask a question:', placeholder='e.g. Should AI replace jobs?')

c1, c2, c3 = st.columns(3)
if c1.button('Should AI replace jobs?'):  question = 'Should AI replace jobs?'
if c2.button('Is crypto good?'):          question = 'Is cryptocurrency good for the economy?'
if c3.button('Allow gene editing?'):      question = 'Should human gene editing be allowed?'

api_key = ""
if st.button('🚀 Get Perspectives', use_container_width=True):
    if not api_key:
        st.error('Please add your Groq API key in the sidebar.')
    elif not question.strip():
        st.warning('Please enter a question.')
    else:
        client = Groq(api_key=api_key)
        sess   = load_onnx_session()
        ranked = rank_perspectives(sess, question)
        st.markdown(f'### 💬 *"{question}"*')
        st.markdown('---')
        icons  = {'Scientist':'🔬','Economist':'📈','Ethicist':'⚖️'}
        colors = {'Scientist':'#00c9a7','Economist':'#ffd166','Ethicist':'#ef476f'}
        for perspective, conf in ranked:
            with st.spinner(f"{icons[perspective]} Asking the {perspective}..."):
                answer = ask_groq(client, question, perspective)
            c = colors[perspective]
            st.markdown(f'<div class="card" style="border-color:{c}"><div class="label" style="color:{c}">{icons[perspective]} {perspective} · {int(conf*100)}% match</div><div class="body">{answer}</div></div>', unsafe_allow_html=True)
"""

with open('app.py', 'w') as f:
    f.write(app_code)
print('✅ app.py written!')

✅ app.py written!


In [ ]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
import onnxruntime as ort
import io
from groq import Groq

api_key = ""  # ← your key here

class SimpleRouter(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(3, 3)
    def forward(self, x):
        return self.fc(x)

def get_keyword_scores(text):
    text = text.lower()
    s   = ["data","research","technology","science","evidence","study","experiment"]
    e   = ["money","market","cost","economy","profit","job","trade","price","growth"]
    eth = ["right","wrong","moral","ethical","fair","should","harm","good","value"]
    return np.array([sum(w in text for w in s),
                     sum(w in text for w in e),
                     sum(w in text for w in eth)], dtype=np.float32)

@st.cache_resource(show_spinner=False)
def load_onnx_session():
    model = SimpleRouter()
    X = torch.eye(3)
    y = torch.tensor([0, 1, 2])
    opt = torch.optim.Adam(model.parameters(), lr=0.1)
    for _ in range(200):
        opt.zero_grad()
        nn.CrossEntropyLoss()(model(X), y).backward()
        opt.step()
    buf = io.BytesIO()
    torch.onnx.export(model, torch.zeros(1, 3), buf,
                      input_names=["scores"], output_names=["logits"], opset_version=11)
    buf.seek(0)
    return ort.InferenceSession(buf.read())

def rank_perspectives(sess, query):
    scores = get_keyword_scores(query).reshape(1, -1)
    logits = sess.run(["logits"], {"scores": scores})[0][0]
    probs  = torch.softmax(torch.tensor(logits), dim=0).numpy()
    names  = ["Scientist", "Economist", "Ethicist"]
    return sorted(zip(names, probs), key=lambda x: -x[1])

def ask_groq(client, question, perspective):
    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": f"You are a {perspective}. Answer in 3-4 sentences from a {perspective.lower()} point of view. Be direct."},
            {"role": "user", "content": question},
        ],
        max_tokens=250, temperature=0.7,
    )
    return resp.choices[0].message.content.strip()

st.set_page_config(page_title="MultiMind AI", page_icon="🧠", layout="centered")
st.markdown("""<style>
.card{padding:1.2rem;border-radius:12px;margin-bottom:1rem;border-left:5px solid;background:#1a1a24;}
.label{font-size:0.78rem;font-weight:700;letter-spacing:2px;text-transform:uppercase;margin-bottom:0.5rem;}
.body{font-size:0.95rem;color:#bbb;line-height:1.7;}
</style>""", unsafe_allow_html=True)

st.title("🧠 MultiMind AI")
st.caption("One question — answered by a Scientist, Economist, and Ethicist.")

# ── Example buttons with session state ──
if "question" not in st.session_state:
    st.session_state.question = ""

c1, c2, c3 = st.columns(3)
if c1.button("Should AI replace jobs?"):
    st.session_state.question = "Should AI replace jobs?"
if c2.button("Is crypto good?"):
    st.session_state.question = "Is cryptocurrency good for the economy?"
if c3.button("Allow gene editing?"):
    st.session_state.question = "Should human gene editing be allowed?"

question = st.text_input("Ask a question:",
                          value=st.session_state.question,
                          placeholder="e.g. Should AI replace jobs?")

if st.button("Get Perspectives", use_container_width=True):
    if not question.strip():
        st.warning("Please enter a question.")
    else:
        client = Groq(api_key=api_key)
        sess   = load_onnx_session()
        ranked = rank_perspectives(sess, question)
        st.markdown(f"### Results for: *{question}*")
        st.markdown("---")
        icons  = {"Scientist":"🔬","Economist":"📈","Ethicist":"⚖️"}
        colors = {"Scientist":"#00c9a7","Economist":"#ffd166","Ethicist":"#ef476f"}
        for perspective, conf in ranked:
            with st.spinner(f"Asking the {perspective}..."):
                answer = ask_groq(client, question, perspective)
            c = colors[perspective]
            st.markdown(f'<div class="card" style="border-color:{c}"><div class="label" style="color:{c}">{icons[perspective]} {perspective} - {int(conf*100)}% match</div><div class="body">{answer}</div></div>', unsafe_allow_html=True)

Overwriting app.py


In [ ]:
NGROK_TOKEN = ""  

from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = NGROK_TOKEN
!pkill -f streamlit 2>/dev/null || true
time.sleep(1)

subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501",
                  "--server.headless", "true",
                  "--server.enableCORS", "false"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
tunnel = ngrok.connect(8501)
print(f"🚀 Open this URL: {tunnel.public_url}")

^C
🚀 Open this URL: https://gumminess-baggie-thrive.ngrok-free.dev
